# QIIME 2 Colab: Gut-to-soil axis tutorial

## 0) Runtime prep
If you just connected/reconnected, run this cell first to ensure a clean state.

In [ ]:
%%bash
set -euo pipefail

echo "Python:" $(python --version)
echo "Resetting any old micromamba install markers (safe to ignore errors)"
rm -f ~/.mamba ~/.conda 2>/dev/null || true


## 1) Install micromamba and QIIME 2
This takes a few minutes but is much lighter than full conda in Colab.

In [ ]:
%%bash
set -euo pipefail

# Install micromamba (static binary)
MAMBA_ROOT_PREFIX=/root/micromamba
export MAMBA_ROOT_PREFIX
mkdir -p "$MAMBA_ROOT_PREFIX"
curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /usr/local/bin bin/micromamba --strip-components=1

echo "==> Creating qiime2 (or latest) env from conda-forge, bioconda, and qiime2/label/"
# Use the official release channel (not qiime2-lts)
/usr/local/bin/micromamba create -y -n qiime2 \
  -c conda-forge -c bioconda -c qiime2 \
  q2cli q2-types q2-feature-table q2templates \
  q2-metadata qiime2 q2-demux q2-dada2 q2-feature-classifier|| {
    echo "Primary create failed, attempting cache clean + retry..."
    /usr/local/bin/micromamba clean -a -y || true
    /usr/local/bin/micromamba create -y -n qiime2 \
      -c conda-forge -c bioconda -c qiime2 \
      qiime2 q2-demux q2-dada2 q2-feature-classifier
}

echo "==> Showing QIIME 2 info"
/usr/local/bin/micromamba run -n qiime2 qiime info

## 2) Upload your metadata and artifacts
Upload your **sample metadata TSV** and any `.qza` / `.qzv` files you want to inspect.

**Note:** If your filename contains spaces, that's fine—just reference it exactly.

In [ ]:
!wget -O 'sample-metadata.tsv' \
  'https://gut-to-soil-tutorial.readthedocs.io/en/latest/data/gut-to-soil/sample-metadata.tsv'


In [ ]:
%%bash
MAMBA_ROOT_PREFIX=/root/micromamba
export MAMBA_ROOT_PREFIX
eval "$(/usr/local/bin/micromamba shell hook -s bash)"
micromamba activate qiime2

qiime metadata tabulate \
  --m-input-file sample-metadata.tsv \
  --o-visualization sample-metadata.qzv

In [ ]:
!wget -O 'demux.qza' \
  'https://gut-to-soil-tutorial.readthedocs.io/en/latest/data/gut-to-soil/demux.qza'

In [ ]:
%%bash
MAMBA_ROOT_PREFIX=/root/micromamba
export MAMBA_ROOT_PREFIX
eval "$(/usr/local/bin/micromamba shell hook -s bash)"
micromamba activate qiime2

qiime demux summarize \
  --i-data demux.qza \
  --o-visualization demux.qzv \

In [ ]:
%%bash

MAMBA_ROOT_PREFIX=/root/micromamba
export MAMBA_ROOT_PREFIX

micromamba run -n qiime2 qiime tools export \
  --input-path demux.qzv \
  --output-path demux_export

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# --- Load data ---
demux = pd.read_csv("demux_export/per-sample-fastq-counts.tsv", sep="\t")
meta  = pd.read_csv("sample-metadata.tsv", sep="\t", comment="#")

# --- Merge demux + metadata (your corrected key logic) ---
left_col  = [c for c in demux.columns if "sample" in c.lower()][0]
right_col = [c for c in meta.columns  if "sample" in c.lower()][0]
merged = demux.merge(meta, left_on=left_col, right_on=right_col, how="left")

# --- Find the metadata column that encodes sample type (same detection as before) ---
col = None
for c in merged.columns:
    if c.lower().replace("_","-") in ["sample-type", "sampletype", "environment", "body-site"]:
        col = c
        break
if col is None:
    raise ValueError("Could not find a column for sample type (check metadata headers).")

# --- Compute total reads per sample (paired or single-end) ---
if 'reverse sequence count' in merged.columns:
    merged['total_reads'] = merged[['forward sequence count', 'reverse sequence count']].sum(axis=1)
else:
    merged['total_reads'] = merged['forward sequence count']

# --- Summaries per sample type (adds percentages) ---
summary = (
    merged.groupby(col, dropna=False)
    .agg(
        total_reads=('total_reads', 'sum'),
        num_samples=('total_reads', 'count'),
        mean_reads=('total_reads', 'mean')
    )
    .sort_values('total_reads', ascending=False)
)

# Percent of total reads & percent of samples
summary['pct_reads']   = 100 * summary['total_reads'] / summary['total_reads'].sum()
summary['pct_samples'] = 100 * summary['num_samples'] / summary['num_samples'].sum()

print("\nSummary by sample type:")
print(summary)

# --- Plots ---
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

# Pie (total reads by type)
ax[0].pie(
    summary['total_reads'],
    labels=[f"{idx} ({p:.1f}%)" for idx, p in zip(summary.index, summary['pct_reads'])],
    autopct=None, startangle=90, counterclock=False
)
ax[0].set_title("Proportion of Total Reads by Sample Type")

# Bar (number of samples by type)
bars = ax[1].bar(summary.index.astype(str), summary['num_samples'])
ax[1].set_title("Number of Samples per Type")
ax[1].set_ylabel("Samples")
ax[1].tick_params(axis='x', rotation=45)

# Add value labels on bars
for b in bars:
    ax[1].annotate(f"{int(b.get_height())}",
                   xy=(b.get_x() + b.get_width()/2, b.get_height()),
                   xytext=(0, 3), textcoords="offset points",
                   ha='center', va='bottom')

plt.tight_layout()
plt.show()

# --- Optional: flag low-read samples (same as before) ---
threshold = merged['total_reads'].median() * 0.2
low = merged.loc[merged['total_reads'] < threshold, [left_col, col, 'total_reads']]
if not low.empty:
    print(f"\n⚠️ Samples with <20% of median reads ({threshold:.0f}):")
    print(low)

# --- Optional: export summary to CSV ---
# summary.to_csv("demux_sample_type_summary.csv")

##Run DADA2 to trim and denoise the data

In [ ]:
%%bash

MAMBA_ROOT_PREFIX=/root/micromamba
export MAMBA_ROOT_PREFIX

micromamba run -n qiime2 qiime dada2 denoise-paired \
  --i-demultiplexed-seqs demux.qza \
  --p-trim-left-f 0 \
  --p-trunc-len-f 250 \
  --p-trim-left-r 0 \
  --p-trunc-len-r 250 \
  --o-representative-sequences asv-seqs.qza \
  --o-table asv-table.qza \
  --o-denoising-stats stats.qza

In [ ]:
%%bash

MAMBA_ROOT_PREFIX=/root/micromamba
export MAMBA_ROOT_PREFIX

micromamba run -n qiime2 qiime metadata tabulate \
  --m-input-file stats.qza \
  --o-visualization stats.qzv

In [ ]:
%%bash

MAMBA_ROOT_PREFIX=/root/micromamba
export MAMBA_ROOT_PREFIX

micromamba run -n qiime2 qiime feature-table summarize-plus \
  --i-table asv-table.qza \
  --m-metadata-file sample-metadata.tsv \
  --o-summary asv-table.qzv \
  --o-sample-frequencies sample-frequencies.qza \
  --o-feature-frequencies asv-frequencies.qza

In [ ]:
%%bash

MAMBA_ROOT_PREFIX=/root/micromamba
export MAMBA_ROOT_PREFIX

micromamba run -n qiime2 qiime feature-table tabulate-seqs \
  --i-data asv-seqs.qza \
  --m-metadata-file asv-frequencies.qza \
  --o-visualization asv-seqs.qzv

In [ ]:
%%bash

MAMBA_ROOT_PREFIX=/root/micromamba
export MAMBA_ROOT_PREFIX

micromamba run -n qiime2 qiime feature-table filter-features \
  --i-table asv-table.qza \
  --p-min-samples 2 \
  --o-filtered-table asv-table-ms2.qza

In [ ]:
%%bash

MAMBA_ROOT_PREFIX=/root/micromamba
export MAMBA_ROOT_PREFIX

micromamba run -n qiime2 qiime feature-table filter-seqs \
  --i-data asv-seqs.qza \
  --i-table asv-table-ms2.qza \
  --o-filtered-data asv-seqs-ms2.qza

In [ ]:
%%bash

MAMBA_ROOT_PREFIX=/root/micromamba
export MAMBA_ROOT_PREFIX

micromamba run -n qiime2 qiime feature-table summarize-plus \
  --i-table asv-table-ms2.qza \
  --m-metadata-file sample-metadata.tsv \
  --o-summary asv-table-ms2.qzv \
  --o-sample-frequencies sample-frequencies-ms2.qza \
  --o-feature-frequencies asv-frequencies-ms2.qza

In [ ]:
!wget -O 'suboptimal-16S-rRNA-classifier.qza' \
  'https://gut-to-soil-tutorial.readthedocs.io/en/latest/data/gut-to-soil/suboptimal-16S-rRNA-classifier.qza'

In [ ]:
%%bash

MAMBA_ROOT_PREFIX=/root/micromamba
export MAMBA_ROOT_PREFIX

micromamba run -n qiime2 qiime feature-classifier classify-sklearn \
  --i-classifier suboptimal-16S-rRNA-classifier.qza \
  --i-reads asv-seqs-ms2.qza \
  --o-classification taxonomy.qza

In [ ]:
%%bash

MAMBA_ROOT_PREFIX=/root/micromamba
export MAMBA_ROOT_PREFIX

micromamba run -n qiime2 qiime feature-table tabulate-seqs \
  --i-data asv-seqs-ms2.qza \
  --i-taxonomy Greengenes_13_8:taxonomy.qza \
  --m-metadata-file asv-frequencies-ms2.qza \
  --o-visualization asv-seqs-ms2.qzv

#Non-working boots install

In [ ]:
%%bash
set -euo pipefail

# --- NEW: create the q2-boots environment from the official YAML ---
Q2_BOOTS_ENV="q2-boots-amplicon-2025.7"
YAML_URL="https://raw.githubusercontent.com/caporaso-lab/q2-boots/refs/heads/main/environment-files/q2-boots-qiime2-amplicon-2025.7-release-2025.7.beta.yml"

echo "==> Fetching q2-boots environment YAML"
curl -L "$YAML_URL" -o q2-boots.yml

echo "==> Creating env $Q2_BOOTS_ENV from q2-boots.yml"
# micromamba understands conda YAML files:
# - uses channels specified inside the YAML
# - installs pinned versions required by q2-boots
/usr/local/bin/micromamba create -y -n "$Q2_BOOTS_ENV" -f q2-boots.yml || {
  echo "First attempt failed; cleaning caches and retrying..."
  /usr/local/bin/micromamba clean -a -y || true
  /usr/local/bin/micromamba create -y -n "$Q2_BOOTS_ENV" -f q2-boots.yml
}

echo "==> Sanity checks"
# Show QIIME 2 info for the boots env
/usr/local/bin/micromamba run -n "$Q2_BOOTS_ENV" qiime info | head -n 20

# List plugins to confirm 'boots' is registered
/usr/local/bin/micromamba run -n "$Q2_BOOTS_ENV" qiime info | sed -n '/Installed plugins:/,/^$/p'

# Optional: show boots CLI help
/usr/local/bin/micromamba run -n "$Q2_BOOTS_ENV" qiime boots --help | head -n 25

# --- Convenience wrapper so you can call 'qiime-boots ...' in any %%bash cell ---
cat <<'WRAP' >/usr/local/bin/qiime-boots
#!/usr/bin/env bash
export MAMBA_ROOT_PREFIX=/root/micromamba
exec /usr/local/bin/micromamba run -n q2-boots-amplicon-2025.7 qiime "$@"
WRAP
chmod +x /usr/local/bin/qiime-boots

echo "==> Ready. Use 'qiime-boots ...' for q2-boots commands."


In [ ]:
%%bash

MAMBA_ROOT_PREFIX=/root/micromamba
export MAMBA_ROOT_PREFIX

micromamba run -n qiime-boots qiime q2-boots kmer-diversity \
  --i-table asv-table-ms2.qza \
  --i-sequences asv-seqs-ms2.qza \
  --m-metadata-file sample-metadata.tsv \
  --p-sampling-depth 96 \
  --p-n 10 \
  --p-replacement \
  --p-alpha-average-method median \
  --p-beta-average-method medoid \
  --output-dir boots-kmer-diversity

Skipping boots

#Taxonomic analysis

In [ ]:
%%bash

MAMBA_ROOT_PREFIX=/root/micromamba
export MAMBA_ROOT_PREFIX

micromamba run -n qiime2 qiime taxa barplot \
  --i-table asv-table-ms2.qza \
  --i-taxonomy taxonomy.qza \
  --m-metadata-file sample-metadata.tsv \
  --o-visualization taxa-bar-plots.qzv

In [ ]:
%%bash

MAMBA_ROOT_PREFIX=/root/micromamba
export MAMBA_ROOT_PREFIX

micromamba run -n qiime2 qiime feature-table filter-samples \
  --i-table asv-table-ms2.qza \
  --m-metadata-file sample-metadata.tsv \
  --p-where '[SampleType] IN ("Human Excrement Compost", "Human Excrement", "Food Compost")' \
  --o-filtered-table asv-table-ms2-dominant-sample-types.qza

In [ ]:
%%bash

MAMBA_ROOT_PREFIX=/root/micromamba
export MAMBA_ROOT_PREFIX

micromamba run -n qiime2 qiime composition ancombc2 \
  --i-table asv-table-ms2-dominant-sample-types.qza \
  --m-metadata-file sample-metadata.tsv \
  --p-fixed-effects-formula SampleType \
  --p-reference-levels 'SampleType::Human Excrement Compost' \
  --o-ancombc2-output ancombc2-results.qza